# Amazon Bedrock API Overview

This notebook demonstrates Bedrock endpoint concepts and common API patterns from the AWS Bedrock certification course. It covers `InvokeModel`, `InvokeModelWithResponseStream`, `StartAsyncInvoke`, and `CreateModelInvocationJob`.

## Bedrock Endpoints

- `bedrock`: control plane for model metadata and administrative actions.
- `bedrock-runtime`: data plane for real-time inference.
- `bedrock-agent`: control plane for agent and prompt flow management.
- `bedrock-agent-runtime`: data plane for agent execution and runtime queries.

In [ ]:
import os
import json
import boto3

region = os.getenv('AWS_REGION', 'us-east-1')
bedrock_runtime = boto3.client('bedrock-runtime', region_name=region)
bedrock = boto3.client('bedrock', region_name=region)

print('Bedrock runtime client configured for', region)

## InvokeModel Example

This example uses `bedrock-runtime` to invoke a model synchronously for immediate output.

In [ ]:
model_id = 'amazon.titan-text-express-v1'
payload = {
    'inputText': 'Explain the key benefits of Amazon Bedrock for a data engineer.',
    'textGenerationConfig': {
        'maxTokenCount': 300,
        'temperature': 0.5,
        'topP': 0.9,
    },
}

response = bedrock_runtime.invoke_model(
    modelId=model_id,
    body=json.dumps(payload),
)
result = json.loads(response['body'].read())
print(json.dumps(result, indent=2))

## InvokeModelWithResponseStream Example

This example demonstrates streaming model output as chunks arrive.

In [ ]:
stream_payload = {
    'inputText': 'Create a short outline for a blog post about Bedrock API endpoints.',
    'textGenerationConfig': {
        'maxTokenCount': 250,
        'temperature': 0.6,
        'topP': 0.95,
    },
}

stream_response = bedrock_runtime.invoke_model_with_response_stream(
    modelId=model_id,
    body=json.dumps(stream_payload),
)

for event in stream_response.get('body', []):
    if 'chunk' in event and 'bytes' in event['chunk']:
        chunk = json.loads(event['chunk']['bytes'])
        print(chunk.get('outputText', ''), end='')
print()

## StartAsyncInvoke Example

Use `StartAsyncInvoke` when you need the model to run in the background and return a job ARN immediately.

In [ ]:
import random

async_payload = {
    'taskType': 'TEXT_VIDEO',
    'textToVideoParams': {'text': 'A futuristic cityscape at sunrise.'},
    'videoGenerationConfig': {
        'fps': 24,
        'durationSeconds': 5,
        'dimension': '1280x720',
        'seed': random.randint(0, 2147483646),
    },
}
output_config = {
    's3OutputDataConfig': {
        's3Uri': 's3://<bucket_name>/<prefix>/',
    },
}

async_response = bedrock_runtime.start_async_invoke(
    modelId='amazon.nova-reel-v1:0',
    modelInput=async_payload,
    outputDataConfig=output_config,
)
print('Async invocation ARN:', async_response.get('invocationArn'))

## CreateModelInvocationJob Example

Use `CreateModelInvocationJob` for batch jobs that process input files from S3 and write results back to S3.

In [ ]:
job_response = bedrock.create_model_invocation_job(
    roleArn='arn:aws:iam::<account>:role/<bedrock-invoke-role>',
    modelId='anthropic.claude-3-haiku-20240307-v1:0',
    jobName='bedrock-batch-demo',
    inputDataConfig={
        's3InputDataConfig': {
            's3Uri': 's3://<bucket_name>/<input_file>.jsonl',
        },
    },
    outputDataConfig={
        's3OutputDataConfig': {
            's3Uri': 's3://<bucket_name>/<output-prefix>/',
        },
    },
)
print('Job ARN:', job_response.get('jobArn'))

## Best Practices

- Use explicit model IDs and avoid generic defaults.
- Keep prompt size and model output length under control.
- Implement retry and exponential backoff for throttling.
- Validate response bodies for the model you are calling.
- Sanitize or redact any sensitive information before logging.